In [1]:
import os
import tensorflow as tf
from datasets import load_dataset
from dotenv import load_dotenv
from transformers import AutoTokenizer

I0000 00:00:1795340168.990258   37953 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795340169.772968   37953 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795340171.724278   37953 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
/home/rayane/.python_env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
load_dotenv()

True

In [3]:
dataset = load_dataset(
    "parquet",
    data_files={
        "train": os.path.join(os.environ["HF_HOME"], "fineweb/datasets--HuggingFaceFW--fineweb/snapshots/9bb295ddab0e05d785b879661af7260fed5140fc/sample/10BT/*.parquet"),
    },
    split="train",
    streaming=True)

In [4]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.add_special_tokens({"pad_token": "<|PAD|>"})

1

In [5]:
for example in dataset.take(1):

    tokenized_text = tokenizer.encode(example["text"])
    split_index = len(tokenized_text) // 2
    encoder_input = tokenized_text[:split_index]
    decoder_output = tokenized_text[split_index:]
    decoder_input = [tokenizer.bos_token_id] + tokenized_text[split_index:-1]
    print(example["text"])

|Viewing Single Post From: Spoilers for the Week of February 11th|
|Lil||Feb 1 2013, 09:58 AM|
Don't care about Chloe/Taniel/Jen-Jen. Don't care about Sami, really, but hoping that we get some good "SAMANTHA GENE!!" Marlena Death-Stares out of it. And "newfound" feelings. Please. If only.
STEFANO!! STEFANO, STEFANO, STEFANO!!!! :cheer:
|Spoilers for the Week of February 11th · DAYS: News, Spoilers & Discussion|


In [6]:
TARGET_LENGTH = 128
SOURCE_LENGTH = 128

def preprocess_data(text):

    tokens = tokenizer.encode(text, add_special_tokens=False)
    tokens.append(tokenizer.eos_token_id)

    assert len(tokens) > 2

    if len(tokens) <= TARGET_LENGTH + SOURCE_LENGTH:

        split_index = len(tokens) // 2
        encoder_input = tokens[:split_index]
        decoder_output = tokens[split_index:]
        decoder_input = [tokenizer.bos_token_id] + tokens[split_index:-1]

        if len(encoder_input) == 0 or len(decoder_output) == 0:
            return

        yield (encoder_input, decoder_input, decoder_output)

    else:

        for i in range(0, len(tokens), TARGET_LENGTH):

            encoder_input = tokens[i:i + TARGET_LENGTH]
            decoder_output = tokens[i + TARGET_LENGTH:i + TARGET_LENGTH + SOURCE_LENGTH]

            if len(encoder_input) == 0 or len(decoder_output) == 0:
                continue

            decoder_input = [tokenizer.bos_token_id] + decoder_output[:-1]

            yield (encoder_input, decoder_input, decoder_output)

In [7]:
def generate_dataset():

    for example in dataset:

        for encoder_input, decoder_input, decoder_output in preprocess_data(example["text"]):

            yield ((encoder_input, decoder_input), decoder_output)

In [8]:
output_signature = (
    (
        tf.TensorSpec(shape=(None,), dtype=tf.int32),
        tf.TensorSpec(shape=(None,), dtype=tf.int32)
    ),
    tf.TensorSpec(shape=(None,), dtype=tf.int32))

train_dataset = tf.data.Dataset.from_generator(generate_dataset, output_signature=output_signature)

E0000 00:00:1795340177.680965   37953 cuda_platform.cc:57] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
=== Source Location Trace: ===
external/xla/xla/stream_executor/cuda/cuda_status.cc:50



In [9]:
train_dataset = train_dataset.shuffle(buffer_size=10000)

In [16]:
def get_length_sequence(inputs, output):

    enc_in, dec_in = inputs

    return tf.maximum(
        tf.shape(enc_in)[0],
        tf.shape(dec_in)[0])

In [17]:
train_dataset = train_dataset.bucket_by_sequence_length(
    element_length_func=get_length_sequence,
    bucket_boundaries=[256, 128, 64],
    bucket_batch_sizes=[16, 32, 64, 8],
    padded_shapes=(([None], [None]), [None]),
    padding_values=((tokenizer.pad_token_id, tokenizer.pad_token_id), tokenizer.pad_token_id),
    drop_remainder=False
)

train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)

In [19]:
for (enc_in, dec_in), dec_out in train_dataset.take(5):
    print(enc_in.shape, dec_in.shape)
    print(dec_out.shape)
    print('\n')

(16, 128) (16, 128)
(16, 128)


(16, 128) (16, 128)
(16, 128)


(16, 128) (16, 128)
(16, 128)


(16, 128) (16, 128)
(16, 128)


(16, 128) (16, 128)
(16, 128)


